# Document search in this workspace

This notebook scans PDF, HTML, DOCX, and TXT files in the Notes_Data folder. Choose a search mode below:

- `keyword`: fast keyword/overlap search
- `semantic`: embedding-based semantic search (requires sentence-transformers)
- `ask`: semantic search plus a local Ollama LLM answer grounded in the retrieved chunks

In [1]:
# This cell finds PDF, HTML, DOCX, and TXT files in the Notes_Data folder.
from pathlib import Path
import importlib
import subprocess
import sys
from IPython.display import display, HTML

can_run = True

# Free-threaded Python kernels (python3.13t.exe) frequently break binary wheels.
if sys.executable.lower().endswith("python3.13t.exe"):
    print("This notebook is running on free-threaded Python (python3.13t.exe).")
    print("Please switch the notebook kernel to a standard interpreter (python.exe), then run this cell again.")
    print("Recommended: project .venv\\Scripts\\python.exe")
    can_run = False


def _check_runtime_deps() -> list[str]:
    missing = []
    checks = [
        ("numpy", "numpy"),
        ("lxml.etree", "lxml"),
        ("docx", "python-docx"),
    ]
    for module_name, package_name in checks:
        try:
            importlib.import_module(module_name)
        except Exception:
            missing.append(package_name)
    return sorted(set(missing))


if can_run:
    missing_packages = _check_runtime_deps()
    if missing_packages:
        print("Missing or broken packages detected:", ", ".join(missing_packages))
        print("Repairing packages in the active kernel environment...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-cache-dir",
            *missing_packages,
        ])
        print("Package repair complete. Restart the kernel, then run this cell again.")
        can_run = False

if can_run:
    sys.path.append(str(Path.cwd()))

    # Import the search functions from the main script.
    from rubber_duck import (
        find_document_files,
        build_index,
        search,
        embed_chunks,
        semantic_search,
        generate_answer,
        build_results_payload,
        DEFAULT_USER_ID,
        DEFAULT_LLM_MODEL,
    )
    import json

    # Point the notebook at the documents folder.
    DATA_DIR = Path('Notes_Data')
    document_paths = find_document_files(DATA_DIR)
    print(f'Found {len(document_paths)} document(s) in {DATA_DIR}')

    if document_paths:
        mode = input("Search mode ('keyword', 'semantic', or 'ask'): ").strip().lower() or 'keyword'
        use_semantic = mode in ('semantic', 'ask')

        # Skip building the keyword index when it won't be used (much faster for large libraries).
        index, documents = build_index(document_paths, build_keyword_index=not use_semantic)
        chunk_embeddings = embed_chunks(documents) if use_semantic else None

        query = input('Enter a keyword or phrase: ').strip()
        if query:
            # Search for the best matching documents and show them in a table.
            results = semantic_search(query, chunk_embeddings, documents, top_k=5) if use_semantic else search(query, index, documents, top_k=5)
            if results:
                rows = []
                for rank, result in enumerate(results, start=1):
                    path = result['path']
                    snippet = result['snippet']
                    metadata = result['metadata']
                    rows.append({
                        'Rank': rank,
                        'Document': path.name,
                        'Chunk': f"{metadata['chunk_index'] + 1}/{metadata['total_chunks']}",
                        'Score': round(result['score'], 3),
                        'Snippet': snippet[:140] + ('...' if len(snippet) > 140 else ''),
                        'Link': f'<a href="file:///{path.resolve()}" target="_blank">Open document</a>'
                    })
                html_rows = ''.join(
                    f"<tr><td>{row['Rank']}</td><td>{row['Document']}</td><td>{row['Chunk']}</td><td>{row['Score']}</td><td>{row['Snippet']}</td><td>{row['Link']}</td></tr>"
                    for row in rows
                )
                display(HTML(f"<table><tr><th>Rank</th><th>Document</th><th>Chunk</th><th>Score</th><th>Snippet</th><th>Link</th></tr>{html_rows}</table>"))

                if mode == 'ask':
                    # Ask a local Ollama model to answer using only the retrieved chunks.
                    print('\nGenerating answer...')
                    answer = generate_answer(query, results, model=DEFAULT_LLM_MODEL)
                    print(f'\nAnswer:\n{answer}')

                # Output the results in the standard JSON format.
                payload = build_results_payload(query, results, user_id=DEFAULT_USER_ID)
                print(json.dumps(payload, indent=4))
            else:
                print('No matches found.')
        else:
            print('No query entered.')
    else:
        print('No documents found. Add some PDF, HTML, DOCX, or TXT files to Notes_Data and run the cell again.')

Found 1842 document(s) in Notes_Data


c:\Users\lance\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model 'all-MiniLM-L6-v2' (first run may download it)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3177.55it/s]


Rank,Document,Chunk,Score,Snippet,Link
1,Bayesian Reasoning and Machine Learning (181115).pdf,10/1878,0.697,"Similarly, there is a desire to control increasingly complex systems, possibly containing many inte...",Open document
2,The Allure of Machine Learning - Azure ML.pdf,3/19,0.66,like Big Data? We are collecting more data than we know what to do with. We stuff it into data warehouses and run it through analytics softw...,Open document
3,ChatGPT Cheat Sheet.pdf,22/23,0.611,wers www.neuralmagic.com 29 7. Expert Prompting You can use several conditions discussed in this cheat sheet to prompt ChatGPT to obtain mor...,Open document
4,NVIDIA Tesla V100 Volta GPU Architecture v1.1 (August 2017).pdf,14/100,0.608,el human intelligence in the field of Artificial Intelligence. Machine Learning is a very popular approach to AI that train systems to learn...,Open document
5,Understanding Deep Learning (2023-02-17).pdf,37/1235,0.575,as blogs for Borealis AI and adapted versions are reproduced here with permission. I am grateful for ...,Open document



Generating answer...

Answer:
Could not reach the local LLM at http://localhost:11434/api/generate: HTTP Error 500: Internal Server Error
Make sure Ollama is installed and running, and that the model is pulled (`ollama pull llama3.2`).
{
    "user_id": "placeholder-user",
    "timestamp": "2026-08-03_08-43-23",
    "query": "what is machine learning?",
    "results": [
        {
            "path": "Notes_Data\\Bayesian Reasoning and Machine Learning (181115).pdf",
            "score": 0.697,
            "Snippet": "Similarly, there is a desire to control increasingly complex systems, possibly containing many inte...",
            "metadata": {
                "source": "Notes_Data\\Bayesian Reasoning and Machine Learning (181115).pdf",
                "file_name": "Bayesian Reasoning and Machine Learning (181115).pdf",
                "file_type": "pdf",
                "chunk_index": 9,
                "total_chunks": 1878
            }
        },
        {
            "path": "Note